# Mechanical property distributions
Evolution of B, G, Δz, and φ distributions of jammed states over training.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

H5_PATH = "campaign_analysis.h5"
plt.rcParams.update({'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})

f = h5py.File(H5_PATH, 'r')
m = f['mechanics']
rounds = m['rounds'][:]

KEYS   = ['B', 'G', 'dz', 'phi']
LABELS = ['Bulk modulus B', 'Shear modulus G', 'Excess coordination Δz', 'Packing fraction φ']

print(f"Mechanics: {len(rounds)} sampled rounds")

In [ ]:
# ---- waterfall heatmaps for each mechanical quantity ----
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, k, label in zip(axes.ravel(), KEYS, LABELS):
    edges  = m[f'{k}_hist_edges'][:]
    counts = m[f'{k}_counts'][:]       # (S, M)
    im = ax.imshow(
        counts, aspect='auto', origin='lower',
        extent=[edges[0], edges[-1], rounds[0], rounds[-1]],
        cmap='viridis', interpolation='nearest'
    )
    plt.colorbar(im, ax=ax, label='normalised density')
    ax.set_xlabel(label)
    ax.set_ylabel('training round')
    ax.set_title(f'{label} distribution over training')

fig.tight_layout()
fig.savefig('mechanical_waterfalls.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- mean ± std fill-between for all four quantities ----
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, k, label in zip(axes.ravel(), KEYS, LABELS):
    mu  = m[f'{k}_mean'][:]
    sig = m[f'{k}_std'][:]
    ax.fill_between(rounds, mu - sig, mu + sig, alpha=0.25)
    ax.plot(rounds, mu, lw=1.5, label='mean')
    ax.set_xlabel('round'); ax.set_ylabel(label)
    ax.set_title(f'{label} mean ± std')

fig.tight_layout()
fig.savefig('mechanical_mean_std.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- correlation: phi_mean vs B_mean and phi_mean vs G_mean, coloured by round ----
phi_mean = m['phi_mean'][:]
B_mean   = m['B_mean'][:]
G_mean   = m['G_mean'][:]

cmap = cm.get_cmap('viridis')
norm = plt.Normalize(rounds.min(), rounds.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
sc1 = ax1.scatter(phi_mean, B_mean, c=rounds, cmap=cmap, norm=norm, s=20, alpha=0.8)
ax1.set_xlabel('⟨φ⟩'); ax1.set_ylabel('⟨B⟩')
ax1.set_title('Bulk modulus vs packing fraction')
plt.colorbar(sc1, ax=ax1, label='round')

sc2 = ax2.scatter(phi_mean, G_mean, c=rounds, cmap=cmap, norm=norm, s=20, alpha=0.8)
ax2.set_xlabel('⟨φ⟩'); ax2.set_ylabel('⟨G⟩')
ax2.set_title('Shear modulus vs packing fraction')
plt.colorbar(sc2, ax=ax2, label='round')

fig.tight_layout()
fig.savefig('mechanical_correlations.pdf', bbox_inches='tight')
plt.show()